Get scalers data for article

# Libraries

In [12]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

from pathlib import Path
import sys

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling3_utils import (
    # _prepare_df,
    MLPipeline,
    _create_pipeline,
    StatsModelsEstimator,
)

# Splashing

In [13]:
pipe_spl = joblib.load('../results/models_modelling3/Logit_splashing_Stk_diam')
pipe_spl

Pipeline(steps=[('init_transformer',
                 InitialTransformer(log_features=('relative_roughness',
                                                  'sedimentation_Stk',
                                                  'particle_droplet_diameter_ratio'))),
                ('column_transformer',
                 ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                                  ('inclination',
                                                   'volume_fraction')),
                                                 ('std', StandardScaler(),
                                                  ('sedimentation_Stk',
                                                   'particle_droplet_diameter_ratio',
                                                   'relative_...),
                                                 ('passthrough', 'passthrough',
                                                  ('wettability',))])),
                ('df_transformer',
                 DataFrameTransformer(add_const=False,
                                      feature_names=['inclination',
                                                     'volume_fraction',
                                                     'sedimentation_Stk',
                                                     'particle_droplet_diameter_ratio',
                                                     'relative_roughness', 'K',
                                                     'wettability'])),
                ('estimator',
                 StatsModelsEstimator(model_class=<class 'statsmodels.discrete.discrete_model.Logit'>))])

In [14]:
transformers = pipe_spl.named_steps['column_transformer'].transformers_
transformers

[('minmax', MinMaxScaler(), ('inclination', 'volume_fraction')),
 ('std',
  StandardScaler(),
  ('sedimentation_Stk',
   'particle_droplet_diameter_ratio',
   'relative_roughness',
   'K')),
 ('passthrough', 'passthrough', ('wettability',)),
 ('remainder', 'drop', [0, 2, 3, 4, 6, 8, 12, 13])]

## MinMaxScaler

In [16]:
min_max_features = transformers[0][2]
min_values = transformers[0][1].data_min_
max_values = transformers[0][1].data_max_
df_minmax = pd.DataFrame({'feature': min_max_features, 'min': min_values, 'max': max_values}).round(2)
df_minmax

,feature,min,max
0,inclination,0.00,0.79
1,volume_fraction,0.03,0.24


In [17]:
print(df_minmax.to_latex())

\begin{tabular}{llrr}
\toprule
 & feature & min & max \\
\midrule
0 & inclination & 0.000000 & 0.790000 \\
1 & volume_fraction & 0.030000 & 0.240000 \\
\bottomrule
\end{tabular}



# StdScaler

In [5]:
std_features = transformers[1][2]
mean_values = transformers[1][1].mean_
std_values = transformers[1][1].scale_
df_spl = pd.DataFrame({'log_10(feature)': std_features, 'mean': mean_values, 'std': std_values}).round(3)
df_spl

,log_10(feature),mean,std
0,sedimentation_Stk,-9.201,3.255
1,particle_droplet_diameter_ratio,-1.712,0.294
2,relative_roughness,-3.962,0.918
3,K,151.512,75.668


# No fragmentation

In [6]:
pipe_nofrgm = joblib.load('../results/models_modelling3/Logit_no_fragmentation_Stk_diam')
pipe_nofrgm

Pipeline(steps=[('init_transformer',
                 InitialTransformer(log_features=('relative_roughness',
                                                  'sedimentation_Stk',
                                                  'particle_droplet_diameter_ratio'))),
                ('column_transformer',
                 ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                                 ('std', StandardScaler(),
                                                  ('sedimentation_Stk',
                                                   'particle_droplet_diameter_ratio',
                                                   'relative_roughness', 'K')),
                                                 ('passthrough'...
                                                  ('wettability', 'inclination',
                                                   'volume_fraction'))])),
                ('df_transformer',
                 DataFrameTransformer(add_const=False,
                                      feature_names=['sedimentation_Stk',
                                                     'particle_droplet_diameter_ratio',
                                                     'relative_roughness', 'K',
                                                     'wettability',
                                                     'inclination',
                                                     'volume_fraction'])),
                ('estimator',
                 StatsModelsEstimator(model_class=<class 'statsmodels.discrete.discrete_model.Logit'>))])

In [7]:
transformers = pipe_nofrgm.named_steps['column_transformer'].transformers_
transformers

[('minmax', MinMaxScaler(), ()),
 ('std',
  StandardScaler(),
  ('sedimentation_Stk',
   'particle_droplet_diameter_ratio',
   'relative_roughness',
   'K')),
 ('passthrough',
  'passthrough',
  ('wettability', 'inclination', 'volume_fraction')),
 ('remainder', 'drop', [0, 2, 3, 4, 6, 8, 12, 13])]

## MinMaxScaler

In [8]:
min_max_features = transformers[0][2]
# min_values = transformers[0][1].data_min_
# max_values = transformers[0][1].data_max_
# pd.DataFrame({'feature': min_max_features, 'min': min_values, 'max': max_values}).round(3)

# StdScaler

In [9]:
std_features = transformers[1][2]
mean_values = transformers[1][1].mean_
std_values = transformers[1][1].scale_
df_nofrgm = pd.DataFrame({'log_10(feature)': std_features, 'mean': mean_values, 'std': std_values}).round(3)
df_nofrgm

,log_10(feature),mean,std
0,sedimentation_Stk,-9.114,3.181
1,particle_droplet_diameter_ratio,-1.723,0.291
2,relative_roughness,-3.946,0.922
3,K,150.526,71.351


# Mean values for std

In [10]:
str_col = df_spl.iloc[:, 0]

# Calculate the mean of the numerical columns
num_cols_mean = (df_spl.iloc[:, 1:] + df_nofrgm.iloc[:, 1:]) / 2

# Combine the string column and the mean of the numerical columns
df_mean = pd.concat([str_col, num_cols_mean], axis=1)
df_mean.round(2)

,log_10(feature),mean,std
0,sedimentation_Stk,-9.16,3.22
1,particle_droplet_diameter_ratio,-1.72,0.29
2,relative_roughness,-3.95,0.92
3,K,151.02,73.51


In [11]:
print(df_mean.round(2).to_latex())

\begin{tabular}{llrr}
\toprule
 & log_10(feature) & mean & std \\
\midrule
0 & sedimentation_Stk & -9.160000 & 3.220000 \\
1 & particle_droplet_diameter_ratio & -1.720000 & 0.290000 \\
2 & relative_roughness & -3.950000 & 0.920000 \\
3 & K & 151.020000 & 73.510000 \\
\bottomrule
\end{tabular}

